In [7]:
import sys
import os

if not os.path.exists("config.py"):
    os.chdir("backend") if os.path.exists("backend") else os.chdir("..")

sys.path.insert(0, os.getcwd())

print("Working directory:", os.getcwd())
print("config.py exists:", os.path.exists("config.py"))

Working directory: c:\Users\Dell\Documents\repo\llms\document-assistant\backend
config.py exists: True


In [11]:
import io
import fitz  # pymupdf
import pdfplumber
from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials
from googleapiclient.http import MediaIoBaseDownload
from config import settings

SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
creds = Credentials.from_authorized_user_file(settings.google_token_file, SCOPES)
service = build("drive", "v3", credentials=creds)

SCOPE_FILE_ID = "1zZkXUA5Hu4fgi7xGq4Ql_U3eoYSdbgdJ"

# Download
request = service.files().get_media(fileId=SCOPE_FILE_ID)
buffer = io.BytesIO()
downloader = MediaIoBaseDownload(buffer, request)
done = False
while not done:
    _, done = downloader.next_chunk()
pdf_bytes = buffer.getvalue()

# Compare page 15 (likely has bullets)
PAGE = 14  # 0-indexed

print("=== pdfplumber ===")
with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
    print(repr(pdf.pages[PAGE].extract_text()[:1000]))

print("\n=== pymupdf ===")
doc = fitz.open(stream=pdf_bytes, filetype="pdf")
print(repr(doc[PAGE].get_text()[:1000]))

=== pdfplumber ===
'STATE OF QATAR SCHEDULE A: PROJECT BRIEF\nASHGHAL\n\uf0b7 The Scope of Work doesn’t cover the BeIN building.\n\uf0b7 Any building extensions that were not part of the original construction shall\nbe demolished to restore the Old TV Building to its historical configuration.\nContractor to conduct a survey for the TV building and to identify all the\nextension sections. All the extended areas (Porta cabins) intended to be\ndemolished, must be confirmed and approved by the End-user prior to the\nexecution of works. The prayer room and the technical store highlighted in\nthe plan below is intended to be demolished.\n\uf0b7 Re-arrangement of rooms within the TV building on each floor, in terms of a\ncommon design for all offices, WC and Pantries. The purpose is to unify the\ninterior design, color and material scheme and re-zoning of each floor as\nfollows:\na. Ground Floor: This floor shall include various zones, the Main Reception\nwith a Waiting Area, Technical Office

In [10]:
from services.extractor import get_drive_service
from services.drive_sync import list_drive_files

service = get_drive_service()
files = list_drive_files(service)
for f in files:
    print(f"{f['filename']} — {f['drive_file_id']}")

Scope of Works.pdf — 1zZkXUA5Hu4fgi7xGq4Ql_U3eoYSdbgdJ
Contract C2024-49.pdf — 1c-18lHMiW-wxFnUlhd6y21RNlP6ex_Ek
